Data Ingestion

In [1]:
import os
import requests
import pandas as pd

API_KEY = '981F3F41-D0C0-11F1-9E30-4201AC1DC129'

url = "https://api.purpleair.com/v1/sensors"

headers = {
    "X-API-Key": API_KEY    
}

url = "https://api.purpleair.com/v1/sensors"

params = {
    "fields": "name,latitude,longitude,location_type,last_seen",
    "nwlng": -74.30,
    "nwlat": 40.92,
    "selng": -73.65,
    "selat": 40.48,
    "location_type": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

response.raise_for_status()

data = response.json()

df_sensors = pd.DataFrame(
    data["data"],
    columns=data["fields"]
)

min_lat = 40.49
max_lat = 40.92

min_lon = -74.26
max_lon = -73.70

df_sensors = df_sensors[
    df_sensors["latitude"].between(min_lat, max_lat) &
    df_sensors["longitude"].between(min_lon, max_lon)
].copy()

df_sensors.head()


,sensor_index,last_seen,name,location_type,latitude,longitude
0,4803,1794147160,West 90th Street near CPW,0,40.787937,-73.96807
1,269010,1794182314,Clinton Ave Firehouse,0,40.721570,-74.19295
2,269040,1794182264,Club Eclipse,0,40.715973,-74.22197
3,271321,1794182270,RWJ Fitness & Wellness: Carteret,0,40.578370,-74.21960
5,276266,1794182293,Crown St bet Bedford & Rogers Aves,0,40.666515,-73.95539


In [2]:
import requests
import pandas as pd
import time

# Function that gets sensor data (specified by sensor_id) from time stamp start to end
def get_sensor_history(sensor_id, start, end):
    url = f"https://api.purpleair.com/v1/sensors/{sensor_id}/history"

    params = {
        "start_timestamp": start,
        "end_timestamp": end,
        "average": 1440,
        "fields": "pm2.5_atm"
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )

    response.raise_for_status()
    history = response.json()

    print("Status:", response.status_code)
    print("Fields:", data.get("fields"))
    print("Sensor rows:", len(data.get("data") or []))
    print("DataFrame shape:", df_sensors.shape)

    if not history.get("data"):
        return pd.DataFrame()

    df = pd.DataFrame(
        history["data"],
        columns=history["fields"]
    )

    df["sensor_index"] = sensor_id

    return df


all_data = []

# Get data for each sensor from 2016-2026
for sensor_id in df_sensors["sensor_index"]: # DON'T EXECUTE YET
    try:
        for year in range(2016, 2027):
            df_temp = get_sensor_history(
                int(sensor_id),
                f"{year}-01-01T00:00:00Z",
                f"{year + 1}-01-01T00:00:00Z"
            )
            
            if not df_temp.empty:
               all_data.append(df_temp)
               print(f"Sensor {sensor_id}: {len(df_temp)} rows")
            else:
               print(f"Sensor {sensor_id}: No data")

    except requests.RequestException as e:
        print(f"Sensor {sensor_id}: Error - {e}")
        break

    # time.sleep(0.5)

df_history = (
    pd.concat(all_data, ignore_index=True)
    if all_data else pd.DataFrame()
)

df_history = df_history.dropna(subset=["pm2.5_atm"])

df_history['time_stamp'] = pd.to_datetime(df_history['time_stamp'])
df_history['Month'] = df_history['time_stamp'].dt.month 
df_history['Year'] = df_history['time_stamp'].dt.year

min_lat = 40.49
max_lat = 40.92

min_lon = -74.26
max_lon = -73.70

df_history = df_history[
    df_history["latitude"].between(min_lat, max_lat) &
    df_history["longitude"].between(min_lon, max_lon)
].copy()


Status: 200
Fields: ['sensor_index', 'last_seen', 'name', 'location_type', 'latitude', 'longitude']
Sensor rows: 92
DataFrame shape: (86, 6)
Sensor 4803: No data
Status: 200
Fields: ['sensor_index', 'last_seen', 'name', 'location_type', 'latitude', 'longitude']
Sensor rows: 92
DataFrame shape: (86, 6)
Sensor 4803: 8 rows
Status: 200
Fields: ['sensor_index', 'last_seen', 'name', 'location_type', 'latitude', 'longitude']
Sensor rows: 92
DataFrame shape: (86, 6)
Sensor 4803: 361 rows
Status: 200
Fields: ['sensor_index', 'last_seen', 'name', 'location_type', 'latitude', 'longitude']
Sensor rows: 92
DataFrame shape: (86, 6)
Sensor 4803: 29 rows
Status: 200
Fields: ['sensor_index', 'last_seen', 'name', 'location_type', 'latitude', 'longitude']
Sensor rows: 92
DataFrame shape: (86, 6)
Sensor 4803: No data
Status: 200
Fields: ['sensor_index', 'last_seen', 'name', 'location_type', 'latitude', 'longitude']
Sensor rows: 92
DataFrame shape: (86, 6)
Sensor 4803: No data
Status: 200
Fields: ['sensor

KeyboardInterrupt: 

,time_stamp,pm2.5_atm,sensor_index,Month,Year
0,2017-12-30 00:00:00+00:00,15.9,4803,12,2017
1,2017-12-25 00:00:00+00:00,7.4,4803,12,2017
2,2017-12-29 00:00:00+00:00,5.5,4803,12,2017
3,2017-12-26 00:00:00+00:00,5.6,4803,12,2017
4,2017-11-23 00:00:00+00:00,9.7,4803,11,2017
5,2017-12-28 00:00:00+00:00,4.1,4803,12,2017
6,2017-12-31 00:00:00+00:00,17.3,4803,12,2017
7,2017-12-27 00:00:00+00:00,10.7,4803,12,2017
8,2018-04-29 00:00:00+00:00,8.9,4803,4,2018
9,2018-01-21 00:00:00+00:00,23.0,4803,1,2018


In [ ]:
# We want to merge df_history and df_sensors in order to get latitude and longitude coordinates'

df_history = df_history.merge(
    df_sensors[['sensor_index', 'latitude', 'longitude']],
    on='sensor_index',
    how='inner'
)

df_history = (
    df_history.groupby(
        ['sensor_index', 'Month', 'Year'],
        dropna=False
    )
    .agg({
        'pm2.5_atm': 'mean',
        'latitude': 'first',
        'longitude': 'first',
        'time_stamp': 'first'
    })
    .reset_index()
)

df_history

# Tasks Currently :
# Find a way to get entries for many years
# Split up monthly records

,sensor_index,Month,Year,pm2.5_atm,latitude,longitude,time_stamp
0,4803,1,2018,14.838710,40.787937,-73.96807,2018-01-21 00:00:00+00:00
1,4803,1,2019,9.658621,40.787937,-73.96807,2019-01-17 00:00:00+00:00
2,4803,1,2026,11.051613,40.787937,-73.96807,2026-01-24 00:00:00+00:00
3,4803,2,2018,15.357143,40.787937,-73.96807,2018-02-22 00:00:00+00:00
4,4803,2,2026,12.764286,40.787937,-73.96807,2026-02-07 00:00:00+00:00
...,...,...,...,...,...,...,...
2118,320001,8,2026,11.578947,40.841780,-74.19755,2026-08-20 00:00:00+00:00
2119,320001,9,2026,5.816667,40.841780,-74.19755,2026-09-22 00:00:00+00:00
2120,320001,10,2026,5.196552,40.841780,-74.19755,2026-10-06 00:00:00+00:00
2121,320836,9,2026,6.480000,40.748085,-74.15198,2026-09-26 00:00:00+00:00


In [ ]:
df_history_M = df_history.drop(columns="time_stamp")
df_history_M.to_csv('purple_air_monthlypm25.csv', index=False)

In [ ]:
df_history['sensor_index'].nunique()

67

In [157]:
# Air Data

df = pd.read_csv("epa_data/daily_88101_2016.csv")
for i in range(17, 27):
    df= pd.concat([df, pd.read_csv(f'epa_data/daily_88101_20{str(i)}.csv')], ignore_index=True)

df = df[
    (df['County Code'].isin([5, 47, 61, 81, 85])) & 
    (df['State Code'] == 36)
]      
df = df[['County Code', 'Latitude', 'Longitude', 'Date Local', 'Arithmetic Mean', 'Address', 'State Name', 'Site Num']]
df['Date Local'] = pd.to_datetime(df['Date Local'])
df['Month'] = df['Date Local'].dt.month 
df['Year'] = df['Date Local'].dt.year
df = df.drop_duplicates()

df_monthly = (
    df.groupby(
        ['County Code', 'Month', 'Year', 'Site Num'],
        dropna=False
    )
    .agg({
        'Arithmetic Mean': 'mean',
        'Latitude': 'first',
        'Longitude': 'first',
        'Address': 'first',
        'State Name': 'first',
    })
    .reset_index()
)
borough_map = {
    5: 'Bronx',
    47: 'Brooklyn',
    61: 'Manhattan',
    81: 'Queens',
    85: 'Staten Island'
}

df_monthly['Boroughs'] = df_monthly['County Code'].map(borough_map).fillna('Unknown')
df_monthly

df_monthly.to_csv('epa_output.csv', index=False)

/var/folders/yl/_whlcz951v78mbcsqg64rxd00000gn/T/ipykernel_8657/821518165.py:3: DtypeWarning: Columns (13: Event Type) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("epa_data/daily_88101_2016.csv")
/var/folders/yl/_whlcz951v78mbcsqg64rxd00000gn/T/ipykernel_8657/821518165.py:5: DtypeWarning: Columns (13: Event Type, 27: CBSA Name) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.concat([df, pd.read_csv(f'epa_data/daily_88101_20{str(i)}.csv')], ignore_index=True)
/var/folders/yl/_whlcz951v78mbcsqg64rxd00000gn/T/ipykernel_8657/821518165.py:5: DtypeWarning: Columns (13: Event Type) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.concat([df, pd.read_csv(f'epa_data/daily_88101_20{str(i)}.csv')], ignore_index=True)


In [159]:
df_monthly
len(df_monthly)

1111